<a href="https://colab.research.google.com/github/Alynn-P/nathamon_portfolio/blob/main/weather_data_for_close_loop_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import pandas as pd
import requests
import os
csv_filename = 'weather_udaicharoen_365days.csv'
if os.path.exists(csv_filename):
  df = pd.read_csv(csv_filename)
  df = df[['datetime','precip','temp','windspeed','solarradiation','humidity']]
  df['datetime'] = pd.to_datetime(df['datetime'])
  print(f"success with csv {csv_filename}")
else:
  url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/Udai%20Charoen%2C%20Kuan%20Kalong%2C%20Satun%2C%20Thailand/2025-11-01/2026-10-31?unitGroup=us&include=days&key=Q73BLA7EDAEHG8V2VQV8MHD7E&contentType=json"
  respond = requests.get(url)
  if respond.status_code == 200:
    data = respond.json()
    days_data = data['days']
    df = pd.DataFrame(days_data)
    df = df[['datetime','precip','temp','windspeed','solarradiation','humidity']]
    print(f"success with API {len(df)} days")
  else:
    print(f"couldn't connect API: {respond.status_code}")
if 'df' in locals() and not df.empty:
    display(df.head())

success with API 365 days


,datetime,precip,temp,windspeed,solarradiation,humidity
0,2025-11-01,0.068,82.5,7.4,234.4,81.7
1,2025-11-02,0.092,82.3,5.8,203.5,82.6
2,2025-11-03,0.250,81.4,9.9,155.0,84.0
3,2025-11-04,0.190,82.9,8.3,222.3,77.6
4,2025-11-05,0.036,81.5,11.3,239.2,79.0


In [27]:
if 'df' in locals() and not df.empty:
  df_unit = df[['datetime','precip','temp','windspeed','solarradiation','humidity']].rename(columns={
        'datetime' : 'date',
        'precip' : 'rainfall_mm',
        'temp' : 'temp_c(norm)',
        'windspeed' : 'windspeed_meter(norm)',
        'solarradiation' : 'solarradiation_mj(norm)',
        'humidity' : 'humidity_int'
  })
  df_unit['date'] = pd.to_datetime(df_unit['date'])
  df_unit['temp_c(norm)'] = (5/9)*(df_unit['temp_c(norm)']-32)*(1/40)
  df_unit['windspeed_meter(norm)'] = df_unit['windspeed_meter(norm)']*0.44074/10
  df_unit['solarradiation_mj(norm)'] = df_unit['solarradiation_mj(norm)']*0.0864/800
  df_unit['humidity_int'] = df_unit['humidity_int']/100
  display(df_unit.head())
else:
  print("need to check cell 1")

,date,rainfall_mm,temp_c(norm),windspeed_meter(norm),solarradiation_mj(norm),humidity_int
0,2025-11-01,0.068,0.701389,0.326148,0.025315,0.817
1,2025-11-02,0.092,0.698611,0.255629,0.021978,0.826
2,2025-11-03,0.250,0.686111,0.436333,0.016740,0.840
3,2025-11-04,0.190,0.706944,0.365814,0.024008,0.776
4,2025-11-05,0.036,0.687500,0.498036,0.025834,0.790


In [28]:
#Last 2 days
df_unit['precip_d-1'] = df_unit['rainfall_mm'].shift(1)
df_unit['precip_d-2'] = df_unit['rainfall_mm'].shift(2)
df_unit['last_2_days'] = df_unit['precip_d-2']+df_unit['precip_d-1']
#Next 2 days
df_unit['precip_d+1'] = df_unit['rainfall_mm'].shift(-1)
df_unit['precip_d+2'] = df_unit['rainfall_mm'].shift(-2)
df_unit['next_2_days'] = df_unit['precip_d+2']+df_unit['precip_d+1']
#ET proxy
df_unit['ET'] = (0.5*df_unit['temp_c(norm)'])+(0.3*df_unit['windspeed_meter(norm)'])+(0.2*df_unit['solarradiation_mj(norm)'])-(0.4*df_unit['humidity_int'])
#soilfactor
df_unit['soilfactor'] = 0.8
#SMI daily
df_unit['SMI_daily'] = df_unit['rainfall_mm']-(df_unit['ET']*df_unit['soilfactor'])
#Sum 7 dats SMI
df_unit['sum_7_days_SMI'] = df_unit['SMI_daily'].rolling(window=7,min_periods=1).sum()

display(df_unit[['date','rainfall_mm','ET','SMI_daily','sum_7_days_SMI','last_2_days','next_2_days']].head(10))

,date,rainfall_mm,ET,SMI_daily,sum_7_days_SMI,last_2_days,next_2_days
0,2025-11-01,0.068,0.126802,-0.033441,-0.033441,NaN,0.342
1,2025-11-02,0.092,0.099990,0.012008,-0.021433,NaN,0.440
2,2025-11-03,0.250,0.141303,0.136957,0.115524,0.160,0.226
3,2025-11-04,0.190,0.157618,0.063905,0.179429,0.342,0.940
4,2025-11-05,0.036,0.182328,-0.109862,0.069567,0.440,1.645
5,2025-11-06,0.904,0.078799,0.840961,0.910528,0.226,1.705
6,2025-11-07,0.741,0.124534,0.641372,1.551901,0.940,1.067
7,2025-11-08,0.964,0.125390,0.863688,2.449030,1.645,0.308
8,2025-11-09,0.103,0.105033,0.018974,2.455996,1.705,0.930
9,2025-11-10,0.205,0.152538,0.082970,2.402008,1.067,0.934


In [45]:
import numpy as np

cond_golden = (
    (df_unit['last_2_days']<3)&
    (df_unit['sum_7_days_SMI'].between(2,12))&
    (df_unit['ET'].between(0.15,0.35))&
    (df_unit['next_2_days'].between(8,35))
)

cond_good = (
    (df_unit['last_2_days']<3)&
    (df_unit['sum_7_days_SMI'].between(0,12))&
    (df_unit['ET'].between(0.15,0.35))&
    (df_unit['next_2_days'].between(8,35))
)

cond_risky = (
    (df_unit['last_2_days']<5)&
    (df_unit['sum_7_days_SMI'].between(-5,2))&
    (df_unit['next_2_days']<35)
)

cond_avoid = (
    (df_unit['last_2_days']>20)|
    (df_unit['sum_7_days_SMI']<-5)|
    (df_unit['sum_7_days_SMI']>15)|
    (df_unit['next_2_days']>40)
)

conditions = [cond_avoid, cond_risky, cond_golden, cond_good]
choices =['Avoid','Risky','Golden','Good']
df_unit['daily_recommendation'] = np.select(conditions, choices, default='Normal')
display(df_unit[['date','rainfall_mm','last_2_days','sum_7_days_SMI','ET','next_2_days','daily_recommendation']].head(365))

,date,rainfall_mm,last_2_days,sum_7_days_SMI,ET,next_2_days,daily_recommendation
0,2025-11-01,0.068,NaN,-0.033441,0.126802,0.342,Normal
1,2025-11-02,0.092,NaN,-0.021433,0.099990,0.440,Normal
2,2025-11-03,0.250,0.160,0.115524,0.141303,0.226,Risky
3,2025-11-04,0.190,0.342,0.179429,0.157618,0.940,Risky
4,2025-11-05,0.036,0.440,0.069567,0.182328,1.645,Risky
...,...,...,...,...,...,...,...
360,2026-10-27,0.839,2.006,3.808534,0.113765,1.281,Normal
361,2026-10-28,0.729,1.973,4.235727,0.035767,0.568,Normal
362,2026-10-29,0.552,1.568,4.612693,0.081578,0.119,Normal
363,2026-10-30,0.016,1.281,4.173858,0.148673,NaN,Normal


In [41]:
status_count = df_unit['daily_recommendation'].value_counts()
print("test")
display(status_count)

test


,count
daily_recommendation,
Risky,278
Normal,85
Avoid,6


In [31]:
print(df_unit[['last_2_days','sum_7_days_SMI','ET','next_2_days']].describe())

       last_2_days  sum_7_days_SMI          ET  next_2_days
count   363.000000      365.000000  365.000000   363.000000
mean      0.544771        0.939564    0.168390     0.544658
std       1.010266        2.918460    0.058889     1.010311
min       0.000000       -1.544470    0.033949     0.000000
25%       0.050000       -0.620870    0.125967     0.050000
50%       0.268000        0.069567    0.164589     0.268000
75%       0.744500        1.867616    0.205625     0.744500
max      12.171000       21.567316    0.319230    12.171000


In [38]:
idx = 2
print("Date:", df_unit.loc[idx, 'date'])
print("last_2_days (< 5):", df_unit.loc[idx, 'last_2_days'] < 5)
print("sum_7_days_SMI (between -5 and 2):", -5 <= df_unit.loc[idx, 'sum_7_days_SMI'] <= 2)
print("next_2_days (< 35):", df_unit.loc[idx, 'next_2_days'] < 35)
test_val = df_unit.loc[idx, 'sum_7_days_SMI']
print(" ค่า sum_7_days_SMI จริงคือ:", test_val)

Date: 2025-11-03 00:00:00
last_2_days (< 5): True
sum_7_days_SMI (between -5 and 2): True
next_2_days (< 35): True
 ค่า sum_7_days_SMI จริงคือ: 0.11552398755555544
